# 02 — Mutation Rate Analysis: Chr 7 (Parent) vs Derived Copies==============================================================Pairwise divergence analysis to test whether MROH6 copies show elevatedsubstitution rates consistent with RNA-mediated duplication (retrotransposition).Uses the exon 4-15 filtered alignment from Step 01.Usage:  python notebooks/02_mutation_rate.py

In [ ]:
import syssys.path.insert(0, '../scripts')import numpy as npimport pandas as pdimport matplotlibimport matplotlib.pyplot as pltimport seaborn as snsfrom Bio import AlignIOfrom pathlib import Pathfrom scipy import stats as sp_statsfrom utils import (    pairwise_divergence_matrix, count_substitution_types,    jukes_cantor_distance, classify_chrom)PROJECT = Path(__file__).resolve().parent.parentDATA_PROC = PROJECT / 'data' / 'processed'RESULTS = PROJECT / 'results'FIG_DIR = RESULTS / 'figures'TABLE_DIR = RESULTS / 'tables'sns.set_context('notebook')sns.set_style('whitegrid')GENOMIC_BASELINE = 0.03  # typical paralog divergence (Lynch & Conery 2000)

## Summary

In [ ]:
print("=" * 70)print("  STEP 02: MUTATION RATE ANALYSIS")

## Summary

In [ ]:
print("=" * 70)

## 2a. Load alignment

## 2a. Loading trimmed alignment

In [ ]:
print("\n── 2a. Loading trimmed alignment ──")aln_path = DATA_PROC / 'mroh6_aligned_trimmed.fasta'alignment = AlignIO.read(aln_path, 'fasta')n_seqs = len(alignment)aln_len = alignment.get_alignment_length()print(f"  Alignment: {n_seqs} sequences x {aln_len} columns")# Build alignment dictionaryaln_dict = {rec.id: str(rec.seq) for rec in alignment}# Load metadata for chromosome classificationloci_meta = pd.read_csv(DATA_PROC / 'mroh6_loci_table.csv')if 'chrom_class' not in loci_meta.columns:    loci_meta['chrom_class'] = loci_meta['chrom'].astype(str).apply(classify_chrom)# Map sequence IDs to chromosome classid_to_chrom = {}id_to_class = {}for _, row in loci_meta.iterrows():    # Match the FASTA ID format from gene_units_to_fasta    locus_id = row.get('locus_id', row.get('gene_unit_id', ''))    fasta_id = f"gu_{locus_id}_chr{row['chrom']}_{row['start']}_{row['end']}"    id_to_chrom[fasta_id] = str(row['chrom'])    id_to_class[fasta_id] = row['chrom_class']

## 2b. Compute pairwise divergence

## 2b. Computing pairwise divergence matrix

In [ ]:
print("\n── 2b. Computing pairwise divergence matrix ──")print(f"  Computing {n_seqs * (n_seqs - 1) // 2} pairwise comparisons...")names, raw_div, jc_div, ts_tv_mat = pairwise_divergence_matrix(aln_dict)# Extract upper triangle valuestriu_idx = np.triu_indices(len(names), k=1)raw_values = raw_div[triu_idx]jc_values = jc_div[triu_idx]tstv_values = ts_tv_mat[triu_idx]# Clean NaN/Infjc_clean = jc_values[~np.isnan(jc_values)]tstv_clean = tstv_values[np.isfinite(tstv_values)]jc_mean = np.nanmean(jc_clean)jc_median = np.nanmedian(jc_clean)raw_mean = np.nanmean(raw_values)tstv_median = np.nanmedian(tstv_clean)fold_diff = jc_mean / GENOMIC_BASELINE if GENOMIC_BASELINE > 0 else np.nan_, p_value = sp_stats.mannwhitneyu(    jc_clean, [GENOMIC_BASELINE] * min(len(jc_clean), 100),    alternative='greater')print(f"  Raw divergence:  mean={raw_mean:.4f}")print(f"  JC-corrected:    mean={jc_mean:.4f}, median={jc_median:.4f}")print(f"  Ts/Tv ratio:     median={tstv_median:.4f}")print(f"  Genomic baseline: {GENOMIC_BASELINE}")print(f"  Fold elevation:  {fold_diff:.1f}x (p={p_value:.2e})")

## 2c. Chr7 vs Derived comparison

## 2c. Chr7 (ancestral) vs Derived divergence

In [ ]:
print("\n── 2c. Chr7 (ancestral) vs Derived divergence ──")chr7_ids = [n for n in names if id_to_class.get(n) == 'chr7_ancestral']derived_ids = [n for n in names if id_to_class.get(n) != 'chr7_ancestral']print(f"  Chr7 copies: {len(chr7_ids)}")print(f"  Derived copies: {len(derived_ids)}")# Chr7-to-derived divergencechr7_to_derived = []for c7 in chr7_ids:    c7_idx = names.index(c7)    for der in derived_ids:        der_idx = names.index(der)        val = jc_div[c7_idx, der_idx]        if not np.isnan(val):            chr7_to_derived.append(val)# Within-chr7 divergencechr7_within = []for i, c7a in enumerate(chr7_ids):    for c7b in chr7_ids[i+1:]:        val = jc_div[names.index(c7a), names.index(c7b)]        if not np.isnan(val):            chr7_within.append(val)# Within-derived divergencederived_within = []for i, da in enumerate(derived_ids[:100]):  # cap for speed    for db in derived_ids[i+1:i+100]:        val = jc_div[names.index(da), names.index(db)]        if not np.isnan(val):            derived_within.append(val)print(f"  Chr7 → Derived mean JC:  {np.mean(chr7_to_derived):.4f}" if chr7_to_derived else "  No chr7→derived pairs")print(f"  Within Chr7 mean JC:     {np.mean(chr7_within):.4f}" if chr7_within else "  No within-chr7 pairs")print(f"  Within Derived mean JC:  {np.mean(derived_within):.4f}" if derived_within else "  No within-derived pairs")

## Figure 1: Divergence distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))# JC-corrected divergence histogramax = axes[0, 0]ax.hist(jc_clean, bins=50, color='steelblue', edgecolor='white', alpha=0.8)ax.axvline(GENOMIC_BASELINE, color='red', linestyle='--', linewidth=2,           label=f'Baseline ({GENOMIC_BASELINE})')ax.axvline(jc_mean, color='gold', linestyle='--', linewidth=2,           label=f'MROH6 mean ({jc_mean:.3f})')ax.set_xlabel('JC-corrected Divergence')ax.set_ylabel('Count')ax.set_title(f'Pairwise Divergence (JC-corrected)\n{fold_diff:.1f}x above baseline, p={p_value:.1e}')ax.legend()# Ts/Tv distributionax = axes[0, 1]ax.hist(tstv_clean[tstv_clean < 5], bins=40, color='seagreen', edgecolor='white')ax.axvline(0.5, color='gray', linestyle=':', label='Random (0.5)')ax.axvline(tstv_median, color='gold', linestyle='--',           label=f'MROH6 median ({tstv_median:.2f})')ax.set_xlabel('Ts/Tv Ratio')ax.set_ylabel('Count')ax.set_title('Transition/Transversion Ratio')ax.legend()# Chr7 vs Derived box plotax = axes[1, 0]plot_data = []if chr7_within:    plot_data.append(('Within\nChr7', chr7_within))if chr7_to_derived:    plot_data.append(('Chr7 →\nDerived', chr7_to_derived))if derived_within:    plot_data.append(('Within\nDerived', derived_within))if plot_data:    bp = ax.boxplot([d for _, d in plot_data],                    labels=[l for l, _ in plot_data],                    patch_artist=True,                    boxprops=dict(facecolor='steelblue', alpha=0.6))    for patch, color in zip(bp['boxes'], ['crimson', 'gold', 'darkorange']):        patch.set_facecolor(color)        patch.set_alpha(0.6)ax.set_ylabel('JC-corrected Divergence')ax.set_title('Divergence by comparison type')ax.axhline(GENOMIC_BASELINE, color='red', linestyle='--', alpha=0.5)# Divergence heatmap (subsampled)ax = axes[1, 1]n_show = min(50, len(names))show_idx = np.linspace(0, len(names)-1, n_show, dtype=int)sub_matrix = jc_div[np.ix_(show_idx, show_idx)]im = ax.imshow(sub_matrix, cmap='YlOrRd', aspect='auto')plt.colorbar(im, ax=ax, label='JC Divergence')ax.set_title(f'Pairwise Divergence Heatmap ({n_show}-copy subsample)')ax.set_xlabel('Copy Index')ax.set_ylabel('Copy Index')plt.tight_layout()plt.savefig(FIG_DIR / 'mutation_rate_analysis.png', dpi=150, bbox_inches='tight')plt.show()print(f"\n  Saved: {FIG_DIR / 'mutation_rate_analysis.png'}")

## Figure 2: Per-chromosome divergence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))# Per-copy mean divergence by chromosomeper_copy = []for i, name in enumerate(names):    others = [j for j in range(len(names)) if j != i]    mean_div = np.nanmean([jc_div[i, j] for j in others])    chrom = id_to_chrom.get(name, 'unknown')    cls = id_to_class.get(name, 'unknown')    per_copy.append({'name': name, 'chrom': chrom, 'chrom_class': cls,                     'mean_jc_div': mean_div})per_copy_df = pd.DataFrame(per_copy)per_copy_df.to_csv(TABLE_DIR / 'per_copy_divergence.csv', index=False)ax = axes[0]chrom_order = per_copy_df.groupby('chrom')['mean_jc_div'].median().sort_values().indexcolor_map = {'chr7_ancestral': 'crimson', 'macro_derived': 'steelblue',             'micro_derived': 'darkorange', 'sex_chrom': 'mediumpurple'}top_chroms = [c for c in chrom_order if              len(per_copy_df[per_copy_df['chrom'] == c]) >= 2][-15:]sub = per_copy_df[per_copy_df['chrom'].isin(top_chroms)]if len(sub) > 0:    chrom_class_map = sub.groupby('chrom')['chrom_class'].first()    palette = {c: color_map.get(chrom_class_map.get(c, ''), 'gray') for c in top_chroms}    sns.boxplot(data=sub, x='chrom', y='mean_jc_div', order=top_chroms,                palette=palette, ax=ax)    ax.set_xlabel('Chromosome')    ax.set_ylabel('Mean JC Divergence')    ax.set_title('Per-copy mean divergence by chromosome')    ax.tick_params(axis='x', rotation=45)# Chr7 vs Derived scatterax = axes[1]for cls, color in color_map.items():    mask = per_copy_df['chrom_class'] == cls    if mask.any():        ax.scatter(per_copy_df.loc[mask].index,                   per_copy_df.loc[mask, 'mean_jc_div'],                   c=color, alpha=0.5, s=15, label=cls)ax.set_xlabel('Copy index')ax.set_ylabel('Mean JC Divergence from all others')ax.set_title('Per-copy divergence (colored by origin)')ax.legend(fontsize=9)ax.axhline(GENOMIC_BASELINE, color='red', linestyle='--', alpha=0.5)plt.tight_layout()plt.savefig(FIG_DIR / 'chr7_vs_derived_scatter.png', dpi=150, bbox_inches='tight')plt.show()print(f"  Saved: {FIG_DIR / 'chr7_vs_derived_scatter.png'}")

## Save summary table

In [ ]:
summary = pd.DataFrame([    {'Metric': 'Number of sequences', 'Value': n_seqs},    {'Metric': 'Alignment columns', 'Value': aln_len},    {'Metric': 'JC-corrected mean', 'Value': f'{jc_mean:.4f}'},    {'Metric': 'JC-corrected median', 'Value': f'{jc_median:.4f}'},    {'Metric': 'Raw divergence mean', 'Value': f'{raw_mean:.4f}'},    {'Metric': 'Ts/Tv median', 'Value': f'{tstv_median:.4f}'},    {'Metric': 'Genomic baseline', 'Value': f'{GENOMIC_BASELINE}'},    {'Metric': 'Fold elevation', 'Value': f'{fold_diff:.1f}x'},    {'Metric': 'P-value', 'Value': f'{p_value:.2e}'},    {'Metric': 'Chr7 copies', 'Value': len(chr7_ids)},    {'Metric': 'Derived copies', 'Value': len(derived_ids)},    {'Metric': 'Chr7→Derived mean JC', 'Value': f'{np.mean(chr7_to_derived):.4f}' if chr7_to_derived else 'N/A'},    {'Metric': 'Within Chr7 mean JC', 'Value': f'{np.mean(chr7_within):.4f}' if chr7_within else 'N/A'},])summary.to_csv(TABLE_DIR / 'mutation_rate_summary.csv', index=False)# Determine verdictif fold_diff > 3 and p_value < 0.05:    verdict = "ROBUSTLY ELEVATED — supports RNA-mediated hypothesis"elif fold_diff > 1.5 and p_value < 0.05:    verdict = "MODERATELY ELEVATED — proceed with caution"else:    verdict = "NOT ELEVATED — consistent with DNA-mediated duplication"

## Summary

In [ ]:
print("\n" + "=" * 70)print("  MUTATION RATE ANALYSIS SUMMARY")

## Summary

In [ ]:
print("=" * 70)print(f"  JC-corrected mean:   {jc_mean:.4f}")print(f"  Genomic baseline:    {GENOMIC_BASELINE}")print(f"  Fold elevation:      {fold_diff:.1f}x (p={p_value:.2e})")print(f"  Ts/Tv median:        {tstv_median:.4f} (>0.5 = transition bias)")print(f"  Verdict:             {verdict}")print(f"\n  => Proceed to Step 03 (dN/dS analysis)")

## Summary

In [ ]:
print("=" * 70)